# Exploratory Data Analysis (EDA) & Data Storytelling
## Analisis Data Penjualan BBM PT. P

### Latar Belakang
Analisis dilakukan untuk memahami performa penjualan BBM
berdasarkan waktu, provinsi, wilayah regional, dan klasifikasi
franchise.

### Tujuan
1. Menghitung rata-rata revenue bulan Januari di wilayah Jawa–Bali.
2. Mengidentifikasi 10 provinsi dengan pertumbuhan revenue
   tertinggi sepanjang 2021.
3. Mengeksplorasi karakteristik yang berkaitan dengan class.
4. Menemukan pola, peluang, dan potensi masalah dalam data.
5. Merumuskan rekomendasi berdasarkan temuan EDA.

### Pendekatan
Data Understanding → Data Quality → Data Cleaning →
Exploratory Analysis → Insight → Rekomendasi Bisnis.

### Tahap 2 — Persiapan environment dan library

In [ ]:
# Hubungkan Google Drive lalu impor library untuk EDA dan parsing angka.
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tentukan lokasi proyek dan dataset.
BASE_PATH = (
    '/content/drive/MyDrive/Colab_Notebooks/'
    'Sains_Data/Tugas_Mandiri_2'
)

DATASET_PATH = os.path.join(
    BASE_PATH,
    'Dataset',
    'data_franchise.csv'
)

# Atur tampilan grafik agar konsisten.
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

# Pastikan file dataset tersedia sebelum dibaca.
assert os.path.exists(DATASET_PATH), (
    f"Dataset tidak ditemukan: {DATASET_PATH}"
)

print("Environment siap.")
print("Dataset:", DATASET_PATH)


### Tahap 3 — Loading dan pemahaman dataset

In [ ]:
# Baca dataset mentah dan lihat ukuran, tipe data, sampel, serta statistik awal.
df_raw = pd.read_csv(DATASET_PATH)
df = df_raw.copy()

print("Ukuran dataset:", df.shape)
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

display(df.head())
display(df.sample(min(5, len(df)), random_state=42))

print("\nNama kolom:")
print(df.columns.tolist())

print("\nInformasi tipe data:")
df.info()

print("\nStatistik deskriptif:")
display(df.describe(include='all').T)


#### Pemeriksaan granularitas data

In [ ]:
# Periksa jumlah tanggal, ID unik, dan jumlah record setiap bulan.
print("Tanggal unik:", df['month_period'].nunique())
print("ID unik:", df['id'].nunique())

print("\nJumlah periode per ID:")
display(
    df.groupby('id')['month_period']
      .nunique()
      .describe()
)

print("\nJumlah record per bulan:")
period_date = pd.to_datetime(
    df['month_period'],
    errors='coerce'
)

display(
    period_date.dt.to_period('M')
    .value_counts()
    .sort_index()
)


### Tahap 4 — Audit kualitas data (Data Quality Assessment)

#### Pemeriksaan missing values

In [ ]:
# Hitung jumlah dan persentase nilai kosong untuk setiap kolom.
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (
        df.isna().mean() * 100
    ).round(2)
}).sort_values('missing_percent', ascending=False)

display(missing_summary)


#### Pemeriksaan duplikasi

In [ ]:
# Periksa duplikasi baris dan duplikasi kombinasi ID-periode.
print(
    "Duplikasi seluruh baris:",
    df.duplicated().sum()
)

print(
    "Duplikasi kombinasi ID dan tanggal:",
    df.duplicated(
        subset=['id', 'month_period']
    ).sum()
)


#### Pemeriksaan tipe data

In [ ]:
# Lihat tipe data asli sebelum kolom dibersihkan atau dikonversi.
print(df.dtypes)


#### Pemeriksaan nilai tidak wajar

In [ ]:
# Perbaiki pembacaan kolom revenue untuk pemeriksaan kualitas data.
# Pada CSV ini, sebagian nilai revenue menggunakan 5 titik sebagai pemisah digit.
# Untuk pola tersebut, 9 digit paling kanan diperlakukan sebagai bagian desimal.
def parse_revenue_value(value):
    # Pertahankan nilai kosong sebagai NaN.
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    # Nilai dengan nol atau satu titik sudah berbentuk angka biasa.
    if text.count('.') <= 1:
        return pd.to_numeric(text, errors='coerce')

    # Pulihkan format khusus yang ditemukan pada kolom revenue.
    if text.count('.') == 5:
        digits = re.sub(r'\D', '', text)
        if len(digits) > 9:
            number = float(f"{digits[:-9]}.{digits[-9:]}")
            return -number if text.startswith('-') else number

    # Jangan menebak format yang tidak dikenali.
    return np.nan

# Periksa nilai dengan parser tanpa mengubah kolom asli terlebih dahulu.
revenue_check = df['revenue'].map(parse_revenue_value)

print("Ringkasan revenue setelah parsing:")
display(revenue_check.describe())
print("Revenue gagal diparsing:", revenue_check.isna().sum())
print("Revenue negatif:", (revenue_check < 0).sum())
print("Revenue nol:", (revenue_check == 0).sum())

# Tampilkan contoh jika masih ada format revenue yang belum dikenali.
if revenue_check.isna().any():
    display(
        df.loc[revenue_check.isna(), ['id', 'month_period', 'revenue']]
          .head(10)
    )


### Tahap 5 — Data cleaning dan feature engineering

#### Normalisasi nama kolom

In [ ]:
# Rapikan nama kolom agar konsisten menggunakan huruf kecil.
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
)

print(df.columns.tolist())


#### Konversi tanggal

In [ ]:
# Konversi periode menjadi tanggal dan buat fitur tahun serta bulan.
df['month_period'] = pd.to_datetime(
    df['month_period'],
    errors='coerce'
)

df['year'] = df['month_period'].dt.year
df['month'] = df['month_period'].dt.month
df['period'] = df['month_period'].dt.to_period('M')


#### Konversi kolom numerik

In [ ]:
# Konversi revenue menggunakan parser yang telah diperiksa pada tahap audit.
df['revenue'] = df['revenue'].map(parse_revenue_value)

print("Jumlah revenue kosong setelah parsing:", df['revenue'].isna().sum())

# Periksa apakah kolom numerik lain dapat dibaca langsung tanpa mengubah data aslinya.
# Kolom dengan banyak titik memiliki format desimal yang ambigu, jadi jangan dipaksakan.
other_numeric_cols = [
    'revenue_avg',
    'max_avg',
    'revenue_loss',
    'loss_perc'
]

for col in other_numeric_cols:
    if col in df.columns:
        parsed = pd.to_numeric(df[col], errors='coerce')
        success_rate = parsed.notna().mean() * 100
        print(f"{col}: {success_rate:.2f}% nilai dapat dibaca langsung sebagai angka")

        # Ubah kolom lain hanya jika seluruh nilai non-kosong terbaca dengan benar.
        non_missing = df[col].notna()
        if non_missing.sum() > 0 and parsed[non_missing].notna().all():
            df[col] = parsed
        else:
            print(
                f"  {col} dipertahankan sebagai teks karena formatnya perlu diverifikasi."
            )


#### Normalisasi nama provinsi

In [ ]:
# Samakan beberapa variasi penulisan nama provinsi yang diketahui.
province_map = {
    'Nangroe Aceh Darussalam': 'Nanggroe Aceh Darussalam',
    'Dki Jakarta': 'DKI Jakarta',
    'Di Yogyakarta': 'DI Yogyakarta',
    'Jawa barat': 'Jawa Barat',
    'Jawa tengah': 'Jawa Tengah',
    'Jawa timur': 'Jawa Timur'
}

if 'province' in df.columns:
    df['province'] = (
        df['province']
          .astype('string')
          .str.strip()
          .replace(province_map)
    )


#### Menentukan data analisis bulanan

In [ ]:
# Bentuk satu record revenue per franchise dan bulan.
# Data awal menggunakan tanggal hari pertama untuk menandai setiap bulan.
franchise_month = (
    df.groupby(
        ['id', 'province', 'year', 'month'],
        as_index=False
    )
    .agg(revenue=('revenue', 'sum'))
)

print("Ukuran data franchise-bulanan:", franchise_month.shape)


#### Tren revenue bulanan

In [ ]:
# Agregasikan revenue bulanan untuk tiap provinsi dan total perusahaan.
monthly_province = (
    franchise_month
    .groupby(
        ['province', 'year', 'month'],
        as_index=False
    )
    .agg(revenue=('revenue', 'sum'))
)

monthly_company = (
    monthly_province
    .groupby(['year', 'month'], as_index=False)
    .agg(revenue=('revenue', 'sum'))
)

# Tampilkan tren total revenue berdasarkan bulan dan tahun.
plt.figure(figsize=(12, 5))

sns.lineplot(
    data=monthly_company,
    x='month',
    y='revenue',
    hue='year',
    marker='o'
)

plt.title('Tren Revenue Bulanan PT. P')
plt.xlabel('Bulan')
plt.ylabel('Total Revenue')
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()


### Tahap 7 — Analisis revenue Januari di Pulau Jawa–Bali

In [ ]:
# Daftar provinsi yang termasuk wilayah Pulau Jawa–Bali sesuai cakupan tugas.
java_bali = [
    'DKI Jakarta',
    'Banten',
    'Jawa Barat',
    'Jawa Tengah',
    'DI Yogyakarta',
    'Jawa Timur',
    'Bali'
]


In [ ]:
# Filter data Januari 2021 untuk provinsi di wilayah Jawa–Bali.
jan_jb = franchise_month[
    (franchise_month['year'] == 2021) &
    (franchise_month['month'] == 1) &
    (franchise_month['province'].isin(java_bali))
].copy()

print("Jumlah record franchise-bulan:", len(jan_jb))


#### Hitung rata-rata dan ringkasan per provinsi

In [ ]:
# Hitung rata-rata revenue per franchise serta ringkasan total pada setiap provinsi.
avg_revenue_jan_jb = jan_jb['revenue'].mean()

jan_province_summary = (
    jan_jb
    .groupby('province')
    .agg(
        total_revenue=('revenue', 'sum'),
        avg_revenue_per_franchise=('revenue', 'mean'),
        franchise_count=('id', 'nunique')
    )
    .sort_values(
        'avg_revenue_per_franchise',
        ascending=False
    )
)

print(
    "Rata-rata revenue per franchise Jawa-Bali, Januari 2021:",
    avg_revenue_jan_jb
)

display(jan_province_summary.round(2))


### Tahap 8 — Analisis pertumbuhan revenue 10 provinsi tertinggi

#### Siapkan tabel revenue bulanan per provinsi

In [ ]:
# Batasi data ke 2021 lalu siapkan revenue Januari dan Desember untuk perbandingan.
revenue_2021 = monthly_province[
    monthly_province['year'] == 2021
]

revenue_pivot = revenue_2021.pivot(
    index='province',
    columns='month',
    values='revenue'
)

# Pastikan kedua bulan pembanding tersedia dalam dataset.
if 1 not in revenue_pivot.columns or 12 not in revenue_pivot.columns:
    raise ValueError(
        "Data Januari atau Desember 2021 tidak tersedia."
    )

growth_summary = revenue_pivot[[1, 12]].copy()
growth_summary.columns = [
    'revenue_january',
    'revenue_december'
]

# Hitung pertumbuhan nominal dan persentase dari Januari ke Desember.
growth_summary['absolute_growth'] = (
    growth_summary['revenue_december']
    - growth_summary['revenue_january']
)

growth_summary['growth_percent'] = np.where(
    growth_summary['revenue_january'] > 0,
    (
        growth_summary['absolute_growth']
        / growth_summary['revenue_january']
    ) * 100,
    np.nan
)

growth_summary = (
    growth_summary
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['growth_percent'])
    .sort_values('growth_percent', ascending=False)
)

top10_provinces = growth_summary.head(10).reset_index()

display(top10_provinces.round(2))


#### Visualisasi Top 10

In [ ]:
# Bandingkan 10 provinsi teratas berdasarkan persentase pertumbuhan revenue.
plt.figure(figsize=(11, 7))

sns.barplot(
    data=top10_provinces,
    x='growth_percent',
    y='province',
    color='steelblue'  # Tentukan warna eksplisit agar kompatibel dengan versi seaborn.
)

plt.title(
    '10 Provinsi dengan Pertumbuhan Revenue Tertinggi, 2021'
)
plt.xlabel('Pertumbuhan Revenue Januari–Desember (%)')
plt.ylabel('Provinsi')
plt.tight_layout()
plt.show()


### Tahap 9 — Eksplorasi kolom class

#### Distribusi class

In [ ]:
# Hitung jumlah record di setiap kategori class yang benar-benar ada dalam data.
class_counts = df['class'].value_counts(dropna=False)

display(class_counts)

class_counts.plot(
    kind='bar',
    figsize=(8, 5)
)

plt.title('Distribusi Kategori Class')
plt.xlabel('Class')
plt.ylabel('Jumlah Record')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


#### Bandingkan karakteristik numerik setiap class

In [ ]:
# Gunakan hanya kolom yang benar-benar bertipe numerik untuk menghindari ringkasan angka palsu.
class_numeric_candidates = [
    'revenue',
    'revenue_avg',
    'max_avg',
    'revenue_loss',
    'loss_perc'
]

class_numeric_cols = [
    col for col in class_numeric_candidates
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
]

print("Kolom numerik yang digunakan:", class_numeric_cols)

if class_numeric_cols:
    class_summary = (
        df.groupby('class')[class_numeric_cols]
          .agg(['count', 'mean', 'median', 'std'])
    )
    display(class_summary.round(2))
else:
    print("Belum ada kolom numerik yang bisa dibandingkan per class.")


In [ ]:
# Buat boxplot revenue_avg hanya jika kolom tersebut sudah tervalidasi sebagai numerik.
if (
    'revenue_avg' in df.columns
    and pd.api.types.is_numeric_dtype(df['revenue_avg'])
    and df['revenue_avg'].notna().any()
):
    plt.figure(figsize=(9, 5))

    sns.boxplot(
        data=df,
        x='class',
        y='revenue_avg'
    )

    plt.title('Distribusi Rata-rata Revenue Berdasarkan Class')
    plt.xlabel('Class')
    plt.ylabel('Revenue Average')
    plt.tight_layout()
    plt.show()
else:
    print(
        "Boxplot revenue_avg dilewati karena format angkanya belum "
        "bisa dipastikan secara konsisten."
    )


#### Periksa hubungan class dengan regional dan provinsi

In [ ]:
# Bandingkan proporsi setiap class di setiap regional.
if {'regional', 'class'}.issubset(df.columns):
    class_by_region = pd.crosstab(
        df['regional'],
        df['class'],
        normalize='index'
    ) * 100

    display(class_by_region.round(2))

    class_by_region.plot(
        kind='bar',
        stacked=True,
        figsize=(10, 6)
    )

    plt.title('Proporsi Class pada Setiap Regional')
    plt.xlabel('Regional')
    plt.ylabel('Proporsi (%)')
    plt.legend(title='Class')
    plt.tight_layout()
    plt.show()


### Tahap 13 — Ekspor hasil analisis

In [ ]:
# Simpan tabel ringkasan agar mudah digunakan kembali pada presentation deck.
OUTPUT_PATH = os.path.join(
    BASE_PATH,
    'Output'
)

os.makedirs(OUTPUT_PATH, exist_ok=True)

jan_province_summary.to_csv(
    os.path.join(
        OUTPUT_PATH,
        'jawa_bali_january_summary.csv'
    )
)

top10_provinces.to_csv(
    os.path.join(
        OUTPUT_PATH,
        'top10_province_growth_2021.csv'
    ),
    index=False
)

print("Tabel ringkasan tersimpan di:", OUTPUT_PATH)
